In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# model & preprocessing
from sklearn.ensemble import AdaBoostRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# metrics
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error

# 1. custom metrics definition
def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator > 0
    if not np.any(mask): return 0.0
    return np.mean(numerator[mask] / denominator[mask]) * 100

def wape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    numerator = np.sum(np.abs(y_true - y_pred))
    denominator = np.sum(np.abs(y_true))
    if denominator == 0: return np.inf
    return (numerator / denominator) * 100

# visualization setup
sns.set_style("whitegrid")
plt.rc('figure', figsize=(12, 6))

try:
    # 2. load & clean data
    print("loading data...")
    df = pd.read_csv("sales_data.csv")

    # 2.1 initial cleaning
    df['week_start'] = pd.to_datetime(df['week_start'])

    # remove duplicates
    initial_len = len(df)
    df = df.drop_duplicates()
    if len(df) < initial_len:
        print(f"dropped {initial_len - len(df)} duplicates")

    # remove rows with missing essential data
    df = df.dropna(subset=['total_sales', 'seller_id', 'product_category_name'])

    # sort by time
    df = df.sort_values(by=['week_start', 'seller_id']).reset_index(drop=True)

    # 3. outlier removal (iqr method)
    Q1 = df['total_sales'].quantile(0.25)
    Q3 = df['total_sales'].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 3.0 * IQR

    print(f"\noutlier threshold: {upper_bound:.2f}")

    rows_before = len(df)
    df = df[df['total_sales'] <= upper_bound].reset_index(drop=True)

    print(f"outliers removed: {rows_before - len(df)} rows")
    print(f"clean data count: {len(df)} rows")

    # 4. feature engineering (anti-leakage)
    # using historical aggregates (shifted)
    df['seller_avg_sales'] = df.groupby('seller_id')['total_sales'].transform(lambda x: x.shift(1).expanding().mean()).fillna(0)
    df['seller_order_count'] = df.groupby('seller_id')['total_orders'].transform(lambda x: x.shift(1).expanding().sum()).fillna(0)
    df['seller_weeks_active'] = df.groupby('seller_id').cumcount()

    # time features
    df['month'] = df['week_start'].dt.month
    df['week_of_year'] = df['week_start'].dt.isocalendar().week.astype(int)

    # remove cold start
    df = df[df['seller_weeks_active'] > 0].reset_index(drop=True)

    # log transform target
    df['total_sales_log'] = np.log1p(df['total_sales'])

    # 5. preparation & training
    print("\ntraining adaboost regressor...")

    # drop future/leaky columns
    cols_to_drop = [
        'total_sales', 'total_sales_log',
        'week_start', 'seller_city', 'seller_id',
        'total_items', 'total_orders', 'avg_order_value'
    ]

    X = df.drop(cols_to_drop, axis=1)

    numerical_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = ['product_category_name', 'seller_state']
    y = df['total_sales_log']

    # preprocessor pipeline
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ], remainder='drop')

    # model pipeline
    # using n_estimators=100 and learning_rate=0.05 for stability
    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', AdaBoostRegressor(
            n_estimators=100,
            learning_rate=0.05,
            random_state=42
        ))
    ])

    # time-based split
    split_index = int(len(df) * 0.8)
    X_train = X.iloc[:split_index]
    y_train = y.iloc[:split_index]
    X_test = X.iloc[split_index:]
    y_test = y.iloc[split_index:]

    print(f"train size: {len(X_train)}")

    model_pipeline.fit(X_train, y_train)
    print("training done.")

    # 6. evaluation
    log_predictions = model_pipeline.predict(X_test)

    # inverse transform
    y_test_ori = np.expm1(y_test)
    predictions_ori = np.expm1(log_predictions)
    predictions_ori = np.maximum(predictions_ori, 0)

    # metrics
    rmse = np.sqrt(mean_squared_error(y_test_ori, predictions_ori))
    r2 = r2_score(y_test_ori, predictions_ori)
    wape_val = wape(y_test_ori, predictions_ori)

    print(f"\nadaboost results")
    print(f"rmse: {rmse:.2f}")
    print(f"r2 score: {r2:.4f}")
    print(f"wape: {wape_val:.2f}%")

    # visualization
    plt.figure(figsize=(10, 10))
    sns.scatterplot(x=y_test_ori, y=predictions_ori, alpha=0.3, color='orange')

    # perfect prediction line
    limit_min = min(y_test_ori.min(), predictions_ori.min())
    limit_max = max(y_test_ori.max(), predictions_ori.max())
    plt.plot([limit_min, limit_max], [limit_min, limit_max], '--', color='red', linewidth=2, label='perfect prediction')

    plt.title('adaboost: actual vs prediction', fontsize=16)
    plt.xlabel('actual sales', fontsize=12)
    plt.ylabel('predicted sales', fontsize=12)
    plt.legend()
    plt.show()

except Exception as e:
    print(f"error: {e}")